In [13]:
# @title Upload 2 Berkas Excel Baru (1 Pembobot + 1 Bidang Dataset)
from google.colab import files
import os

print("Silakan unggah 2 file Excel:")
print("1. Berkas Pembobot (W8)")
print("2. Berkas Dataset TPT")
uploaded_files = files.upload()

# Menampilkan daftar berkas yang berhasil diunggah
print("\nBerkas yang berhasil diunggah:")
for filename in uploaded_files.keys():
    print(f"- {filename} ({len(uploaded_files[filename])} bytes)")

Silakan unggah 2 file Excel:
1. Berkas Pembobot (W8)
2. Berkas Dataset TPT


Saving data_JarakCentroid_10th_TPT_peng1.xlsx to data_JarakCentroid_10th_TPT_peng1 (1).xlsx
Saving W8_JarakCentroid_10th_peng1.xlsx to W8_JarakCentroid_10th_peng1 (1).xlsx

Berkas yang berhasil diunggah:
- data_JarakCentroid_10th_TPT_peng1 (1).xlsx (20352 bytes)
- W8_JarakCentroid_10th_peng1 (1).xlsx (353334 bytes)


### Konfigurasi Nama File dan Analisis Multi-Dataset
Setelah mengunggah berkas-berkas di atas, silakan sesuaikan nama file pada sel di bawah ini sesuai dengan nama file yang Anda unggah, lalu jalankan untuk mengevaluasi seluruh pembobot spasial-temporal terhadap 4 bidang tersebut secara simultan.

In [4]:
# @title Pemrosesan Lintas Pembobot Terpilih (Periode 10 Tahun)
import pandas as pd
import numpy as np
import os
from google.colab import files

# ==========================================
# KONFIGURASI FILE BARU 10 TAHUN
# ==========================================
W8_FILE_PATH = "W8_JarakCentroid_10th_peng1.xlsx"  # Menggunakan file yang baru diunggah

# Hanya 3 bidang (tanpa Pertanian) untuk periode 10 tahun
datasets_config = {
    "TPT": "data_JarakCentroid_10th_TPT_peng1.xlsx",
}

def fit_eq13_fixed_effect(df_in, Wy):
    """
    Fit: y_it = rho*Wy_it + beta*y_{i,t-1} + mu_i + eps_it
    using within transformation (demean by location).
    """
    dfm = df_in.copy()
    dfm["Wy"] = Wy

    # gunakan hanya baris dengan lag tersedia
    dfm = dfm.dropna(subset=["Y_(t-1)"]).copy()

    y = dfm["Y"].astype(float).to_numpy()
    x1 = dfm["Wy"].astype(float).to_numpy()
    x2 = dfm["Y_(t-1)"].astype(float).to_numpy()
    g  = dfm["loc_id"].to_numpy()

    # within (demean by group)
    y_dm  = y  - pd.Series(y).groupby(g).transform("mean").to_numpy()
    x1_dm = x1 - pd.Series(x1).groupby(g).transform("mean").to_numpy()
    x2_dm = x2 - pd.Series(x2).groupby(g).transform("mean").to_numpy()

    X = np.column_stack([x1_dm, x2_dm])

    # OLS
    XtX = X.T @ X
    Xty = X.T @ y_dm
    coef = np.linalg.solve(XtX, Xty)

    rho_hat, beta_hat = coef[0], coef[1]

    # fitted untuk versi demeaned
    yhat_dm = X @ coef
    resid_dm = y_dm - yhat_dm

    # RMSE dan R2 (within)
    rmse = float(np.sqrt(np.mean(resid_dm**2)))
    ss_res = float(np.sum(resid_dm**2))
    ss_tot = float(np.sum((y_dm - y_dm.mean())**2))
    r2 = 1.0 - ss_res/ss_tot if ss_tot > 0 else np.nan

    return {
        "rho_hat": float(rho_hat),
        "beta_hat": float(beta_hat),
        "rmse_within": rmse,
        "r2_within": r2,
        "n_obs": int(len(dfm))
    }

def compute_Wy_bd_dynamic(df, W_bd):
    """
    W_bd: 35x35, dihitung per tahun secara fleksibel untuk total baris berapapun
    """
    Wy = np.zeros(len(df), dtype=float)
    for yr in sorted(df["Waktu"].unique()):
        idx = df.index[df["Waktu"] == yr].to_numpy()
        y_t = df.loc[idx, "Y"].astype(float).to_numpy().reshape(-1,1)
        if y_t.shape[0] != W_bd.shape[0]:
            raise ValueError(f"Year {yr}: expected {W_bd.shape[0]} rows, got {y_t.shape[0]}")
        Wy[idx] = (W_bd @ y_t).ravel()
    return Wy

# Wbq dan Wbr diproses per tahun (35x35) seperti W_bd
# W_bdst tetap menggunakan ekspansi spasial-temporal 350x350 karena merupakan matriks ST
all_results = []

try:
    print("Membaca matriks pembobot dasar dari file Excel...")
    loaded_W_35x35 = {}
    for sh in ["Wbq", "Wbr", "W_bd"]:
        loaded_W_35x35[sh] = pd.read_excel(W8_FILE_PATH, sheet_name=sh, header=None).to_numpy(dtype=float)

    W_bdst_full = pd.read_excel(W8_FILE_PATH, sheet_name="W_bdst", header=None).to_numpy(dtype=float)

    # Memproses setiap bidang dataset (3 bidang, 10 tahun)
    for nama_bidang, file_path in datasets_config.items():
        if not os.path.exists(file_path):
            print(f"[Peringatan] Berkas untuk bidang '{nama_bidang}' ({file_path}) tidak ditemukan.")
            continue

        print(f"Memproses Bidang: {nama_bidang} ({file_path})...")
        df_bidang = pd.read_excel(file_path)

        # Buat ID lokasi
        loc_order = df_bidang["Lokasi"].drop_duplicates().tolist()
        loc_to_id = {name: i for i, name in enumerate(loc_order)}
        df_bidang["loc_id"] = df_bidang["Lokasi"].map(loc_to_id)

        n_loc = df_bidang["loc_id"].nunique()
        t_years = df_bidang["Waktu"].nunique()
        n_total = len(df_bidang)
        print(f"-> Terdeteksi {n_loc} lokasi x {t_years} tahun = {n_total} baris data.")

        # 1. Jalankan untuk Matriks Wbq, Wbr, dan W_bd secara per-tahun (35x35)
        for sh in ["Wbq", "Wbr", "W_bd"]:
            W_matrix = loaded_W_35x35[sh]
            Wy = compute_Wy_bd_dynamic(df_bidang, W_matrix)
            res = fit_eq13_fixed_effect(df_bidang, Wy)
            res.update({
                "Bidang": nama_bidang,
                "Pembobot": sh,
                "Wy_type": f"{sh} per-year ({n_loc}x{n_loc})"
            })
            all_results.append(res)

        # 2. Jalankan untuk Matriks W_bdst (350x350)
        Wy_bdst = (W_bdst_full @ df_bidang["Y"].astype(float).to_numpy().reshape(-1, 1)).ravel()
        res_bdst = fit_eq13_fixed_effect(df_bidang, Wy_bdst)
        res_bdst.update({
            "Bidang": nama_bidang,
            "Pembobot": "W_bdst",
            "Wy_type": f"W_ST ({n_total}x{n_total})"
        })
        all_results.append(res_bdst)

    # Gabungkan semua hasil dan tampilkan
    if all_results:
        grand_res_df = pd.DataFrame(all_results)
        cols = ["Bidang", "Pembobot", "Wy_type", "rho_hat", "beta_hat", "r2_within", "rmse_within", "n_obs"]
        grand_res_df = grand_res_df[cols].sort_values(["Bidang", "Pembobot"]).reset_index(drop=True)
        display(grand_res_df)

        # Simpan hasil kompilasi ke Excel
        output_all_fields = "Hasil_Simulasi_Gabungan_3_Bidang_10th_Filter.xlsx"
        grand_res_df.to_excel(output_all_fields, index=False)
        print(f"\nHasil kompilasi sukses disimpan ke ’{output_all_fields}’!")
        files.download(output_all_fields)
    else:
        print("Tidak ada data yang diproses.")
except Exception as e:
    print(f"Terjadi kesalahan saat memproses data: {e}")

Membaca matriks pembobot dasar dari file Excel...
Memproses Bidang: TPT (data_JarakCentroid_10th_TPT_peng1.xlsx)...
-> Terdeteksi 35 lokasi x 10 tahun = 350 baris data.


,Bidang,Pembobot,Wy_type,rho_hat,beta_hat,r2_within,rmse_within,n_obs
0,TPT,W_bd,W_bd per-year (35x35),0.848045,0.174475,0.486136,0.697180,315
1,TPT,W_bdst,W_ST (350x350),0.581630,-0.551806,0.883669,0.331718,315
2,TPT,Wbq,Wbq per-year (35x35),0.752022,0.165648,0.431191,0.733507,315
3,TPT,Wbr,Wbr per-year (35x35),0.723737,0.181459,0.422438,0.739129,315



Hasil kompilasi sukses disimpan ke ’Hasil_Simulasi_Gabungan_3_Bidang_10th_Filter.xlsx’!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
# @title Rata-rata Metrik Performa Berdasarkan Tipe Pembobot
import pandas as pd

# Membaca hasil kompilasi dasar jika variabel grand_res_df sudah ada
if 'grand_res_df' in globals():
    df_metrics = grand_res_df.copy()
else:
    df_metrics = pd.read_excel("Hasil_Simulasi_Gabungan_3_Bidang_10th_Filter.xlsx")

# Grouping dan menghitung rata-rata R2 dan RMSE
summary_by_weight = df_metrics.groupby("Pembobot").agg(
    mean_r2_within=("r2_within", "mean"),
    mean_rmse_within=("rmse_within", "mean"),
    total_observations=("n_obs", "sum")
).reset_index()

print("=== Rata-rata Kinerja Model Berdasarkan Tipe Pembobot (3 Bidang) ===")
display(summary_by_weight)

=== Rata-rata Kinerja Model Berdasarkan Tipe Pembobot (3 Bidang) ===


,Pembobot,mean_r2_within,mean_rmse_within,total_observations
0,W_bd,0.486136,0.697180,315
1,W_bdst,0.883669,0.331718,315
2,Wbq,0.431191,0.733507,315
3,Wbr,0.422438,0.739129,315


In [6]:
#2) Wy building function for W_ST (350×350) and W_bd (35×35)

def compute_Wy_ST(df, W_ST):
    """
    W_ST: 350x350, dikalikan langsung dengan vektor y (panjang 350)
    sesuai urutan baris df.
    """
    y_vec = df["Y"].astype(float).to_numpy().reshape(-1,1)  # 350x1
    Wy = (W_ST @ y_vec).ravel()
    return Wy

def compute_Wy_bd(df, W_bd):
    """
    W_bd: 35x35, dihitung per tahun:
    Wy_t = W_bd @ y_t, lalu distack sesuai urutan df.
    Asumsi: dalam setiap tahun, urutan 35 lokasi di df sama dengan urutan W_bd.
    """
    Wy = np.zeros(len(df), dtype=float)
    for yr in df["Waktu"].unique():
        idx = df.index[df["Waktu"] == yr].to_numpy()
        y_t = df.loc[idx, "Y"].astype(float).to_numpy().reshape(-1,1)  # 35x1
        if y_t.shape[0] != W_bd.shape[0]:
            raise ValueError(f"Year {yr}: expected {W_bd.shape[0]} rows, got {y_t.shape[0]}")
        Wy[idx] = (W_bd @ y_t).ravel()
    return Wy


In [7]:
#3) Eq.(14) estimation with location fixed effect (within estimator)

def fit_eq13_fixed_effect(df_in, Wy):
    """
    Fit: y_it = rho*Wy_it + beta*y_{i,t-1} + mu_i + eps_it
    using within transformation (demean by location).

    Returns: rho_hat, beta_hat, fitted, residual, R2, RMSE, n_obs
    """
    dfm = df_in.copy()
    dfm["Wy"] = Wy

    # gunakan hanya baris dengan lag tersedia (2019–2022)
    dfm = dfm.dropna(subset=["Y_(t-1)"]).copy()

    y = dfm["Y"].astype(float).to_numpy()
    x1 = dfm["Wy"].astype(float).to_numpy()
    x2 = dfm["Y_(t-1)"].astype(float).to_numpy()
    g  = dfm["loc_id"].to_numpy()

    # within (demean by group)
    y_dm  = y  - pd.Series(y).groupby(g).transform("mean").to_numpy()
    x1_dm = x1 - pd.Series(x1).groupby(g).transform("mean").to_numpy()
    x2_dm = x2 - pd.Series(x2).groupby(g).transform("mean").to_numpy()

    X = np.column_stack([x1_dm, x2_dm])  # tanpa intercept (FE sudah dihapus via demeaning)

    # OLS
    XtX = X.T @ X
    Xty = X.T @ y_dm
    coef = np.linalg.solve(XtX, Xty)

    rho_hat, beta_hat = coef[0], coef[1]

    # fitted untuk versi demeaned
    yhat_dm = X @ coef
    resid_dm = y_dm - yhat_dm

    # RMSE dan R2 (within)
    rmse = float(np.sqrt(np.mean(resid_dm**2)))
    ss_res = float(np.sum(resid_dm**2))
    ss_tot = float(np.sum((y_dm - y_dm.mean())**2))
    r2 = 1.0 - ss_res/ss_tot if ss_tot > 0 else np.nan

    return {
        "rho_hat": float(rho_hat),
        "beta_hat": float(beta_hat),
        "rmse_within": rmse,
        "r2_within": r2,
        "n_obs": int(len(dfm))
    }


In [10]:
#4) Run for all weights
import pandas as pd
import numpy as np
import os

# Menggunakan nama file baru hasil unggahan
W8_PATH = "W8_JarakCentroid_10th_peng1.xlsx"

# Konfigurasi berkas untuk masing-masing bidang sesuai file yang diunggah
datasets_config = {
    "TPT": "data_JarakCentroid_10th_TPT_peng1.xlsx",
}

if not os.path.exists(W8_PATH):
    raise FileNotFoundError(f"File pembobot {W8_PATH} tidak ditemukan. Silakan periksa kembali file yang Anda unggah.")

xls = pd.ExcelFile(W8_PATH)
print("Sheets yang ditemukan di file pembobot:", xls.sheet_names)

# Memilih sheet yang ada di file W8_JarakCentroid_10th_peng1 (1).xlsx
W_ST_sheets = ["Wbq", "Wbr", "W_bdst"]
W_bd_sheet  = "W_bd"

# Membaca seluruh matriks pembobot terlebih dahulu
loaded_W_ST = {}
for sh in W_ST_sheets:
    loaded_W_ST[sh] = pd.read_excel(W8_PATH, sheet_name=sh, header=None).to_numpy(dtype=float)
W_bd = pd.read_excel(W8_PATH, sheet_name=W_bd_sheet, header=None).to_numpy(dtype=float)

all_results = []

# Memulai pemrosesan untuk setiap dataset
for nama_bidang, data_file in datasets_config.items():
    if not os.path.exists(data_file):
        print(f"[Peringatan] Berkas {data_file} untuk {nama_bidang} tidak ditemukan, dilewati.")
        continue

    print(f"Memproses Bidang: {nama_bidang} ({data_file})...")
    df_bidang = pd.read_excel(data_file)

    # Menyiapkan ID lokasi
    loc_order = df_bidang["Lokasi"].drop_duplicates().tolist()
    loc_to_id = {name: i for i, name in enumerate(loc_order)}
    df_bidang["loc_id"] = df_bidang["Lokasi"].map(loc_to_id)

    n = df_bidang["loc_id"].nunique()
    t_years = df_bidang["Waktu"].nunique()

    # 1. Jalankan untuk matriks spasiotemporal
    for sh in W_ST_sheets:
        W_ST = loaded_W_ST[sh]

        # Jika matriks berukuran 35x35 (seperti Wbq dan Wbr), lakukan ekspansi dinamis terlebih dahulu
        if W_ST.shape == (n, n):
            Wy = compute_Wy_bd_dynamic(df_bidang, W_ST)
            wy_type_str = f"{sh} per-year ({n}x{n})"
        else:
            assert W_ST.shape == (len(df_bidang), len(df_bidang)), (sh, W_ST.shape)
            Wy = compute_Wy_ST(df_bidang, W_ST)
            wy_type_str = f"W_ST ({len(df_bidang)}x{len(df_bidang)})"

        out = fit_eq13_fixed_effect(df_bidang, Wy)
        out["Bidang"] = nama_bidang
        out["Weight"] = sh
        out["Wy_type"] = wy_type_str
        all_results.append(out)

    # 2. Jalankan untuk Matriks W_bd
    assert W_bd.shape == (n, n), ("W_bd", W_bd.shape)
    Wy_bd = compute_Wy_bd_dynamic(df_bidang, W_bd)
    out_bd = fit_eq13_fixed_effect(df_bidang, Wy_bd)
    out_bd["Bidang"] = nama_bidang
    out_bd["Weight"] = "W_bd"
    out_bd["Wy_type"] = f"W_bd per-year ({n}x{n})"
    all_results.append(out_bd)

grand_res_df = pd.DataFrame(all_results)
cols = ["Bidang", "Weight", "Wy_type", "rho_hat", "beta_hat", "r2_within", "rmse_within", "n_obs"]
grand_res_df = grand_res_df[cols].sort_values(["Bidang", "Weight"]).reset_index(drop=True)
display(grand_res_df)

Sheets yang ditemukan di file pembobot: ['W_bdst', 'W_bd', 'Wbq', 'Wbr']
Memproses Bidang: TPT (data_JarakCentroid_10th_TPT_peng1.xlsx)...


,Bidang,Weight,Wy_type,rho_hat,beta_hat,r2_within,rmse_within,n_obs
0,TPT,W_bd,W_bd per-year (35x35),0.848045,0.174475,0.486136,0.697180,315
1,TPT,W_bdst,W_ST (350x350),0.581630,-0.551806,0.883669,0.331718,315
2,TPT,Wbq,Wbq per-year (35x35),0.752022,0.165648,0.431191,0.733507,315
3,TPT,Wbr,Wbr per-year (35x35),0.723737,0.181459,0.422438,0.739129,315


In [11]:
#5) Monte Carlo (Block Bootstrap by Location)
### produced 95% CIs for rho, beta as well as RMSE/R2 (within) with resampling per location (block)

def block_bootstrap_eq13(df_in, Wy_full, B=1000, seed=2026):
    rng = np.random.default_rng(seed)
    loc_ids = np.sort(df_in["loc_id"].unique())
    nloc = len(loc_ids)

    stats = []

    for b in range(B):
        # sample lokasi dengan replacement
        samp_locs = rng.choice(loc_ids, size=nloc, replace=True)

        # ambil baris sesuai samp_locs (gabungkan)
        parts = []
        Wy_parts = []
        for loc in samp_locs:
            sub = df_in[df_in["loc_id"] == loc].copy()
            parts.append(sub)
            Wy_parts.append(Wy_full[sub.index.to_numpy()])

        df_b = pd.concat(parts, axis=0)
        Wy_b = np.concatenate(Wy_parts, axis=0)

        try:
            out = fit_eq13_fixed_effect(df_b.reset_index(drop=True), Wy_b)
            stats.append(out)
        except np.linalg.LinAlgError:
            continue

    boot = pd.DataFrame(stats)
    def ci(x):
        return float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5))

    summary = {
        "rho_mean": float(boot["rho_hat"].mean()),
        "rho_ci95": ci(boot["rho_hat"]),
        "beta_mean": float(boot["beta_hat"].mean()),
        "beta_ci95": ci(boot["beta_hat"]),
        "rmse_mean": float(boot["rmse_within"].mean()),
        "rmse_ci95": ci(boot["rmse_within"]),
        "r2_mean": float(boot["r2_within"].mean()),
        "r2_ci95": ci(boot["r2_within"]),
        "B_effective": int(len(boot))
    }
    return summary

# Menggunakan df_bidang (Pendidikan 10th) yang tersedia di kernel
# Mengubah nama sheet dari 'W_ST^Q_diag1' ke sheet valid 'Wbq'
sh = "Wbq"
W_ST_base = pd.read_excel(W8_PATH, sheet_name=sh, header=None).to_numpy(dtype=float)[:35, :35]
t_years = df_bidang["Waktu"].nunique()
W_ST = np.kron(np.eye(t_years), W_ST_base)

y = (W_ST @ df_bidang["Y"].astype(float).to_numpy().reshape(-1, 1)).ravel()
summary = block_bootstrap_eq13(df_bidang, y, B=1000, seed=2026)
summary

{'rho_mean': 0.7567651553353357,
 'rho_ci95': (0.6107317258522336, 0.8884194363197583),
 'beta_mean': 0.16145281528902958,
 'beta_ci95': (0.057424756113226984, 0.2616202809313903),
 'rmse_mean': 0.7301286003874891,
 'rmse_ci95': (0.6322221186980077, 0.8273426080873012),
 'r2_mean': 0.4343066746610809,
 'r2_ci95': (0.3468799810859342, 0.5283496421952363),
 'B_effective': 1000}

In [12]:
# @title Bootstrap untuk Lintas Pembobot Terpilih (Periode 10 Tahun)
import pandas as pd
import numpy as np
import os
from google.colab import files

def block_bootstrap_eq13(df_in, Wy_full, B=1000, seed=2026):
    rng = np.random.default_rng(seed)
    loc_ids = np.sort(df_in["loc_id"].unique())
    nloc = len(loc_ids)

    stats = []

    for b in range(B):
        samp_locs = rng.choice(loc_ids, size=nloc, replace=True)

        parts = []
        Wy_parts = []
        for loc in samp_locs:
            sub = df_in[df_in["loc_id"] == loc].copy()
            parts.append(sub)
            Wy_parts.append(Wy_full[sub.index.to_numpy()])

        df_b = pd.concat(parts, axis=0)
        Wy_b = np.concatenate(Wy_parts, axis=0)

        try:
            out = fit_eq13_fixed_effect(df_b.reset_index(drop=True), Wy_b)
            stats.append(out)
        except np.linalg.LinAlgError:
            continue

    boot = pd.DataFrame(stats)
    def ci(x):
        return float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5))

    summary = {
        "rho_mean": float(boot["rho_hat"].mean()),
        "rho_ci95": ci(boot["rho_hat"]),
        "beta_mean": float(boot["beta_hat"].mean()),
        "beta_ci95": ci(boot["beta_hat"]),
        "rmse_mean": float(boot["rmse_within"].mean()),
        "rmse_ci95": ci(boot["rmse_within"]),
        "r2_mean": float(boot["r2_within"].mean()),
        "r2_ci95": ci(boot["r2_within"]),
        "B_effective": int(len(boot))
    }
    return summary

# Memakai berkas yang diunggah
datasets_config = {
    "TPT": "data_JarakCentroid_10th_TPT_peng1.xlsx",
}

W8_PATH = "W8_JarakCentroid_10th_peng1.xlsx"
W_ST_sheets = ["Wbq", "Wbr", "W_bdst"]
W_bd_sheet  = "W_bd"

# Membaca matriks pembobot terpilih
loaded_W_ST_base = {}
for sh in W_ST_sheets:
    loaded_W_ST_base[sh] = pd.read_excel(W8_PATH, sheet_name=sh, header=None).to_numpy(dtype=float)
W_bd = pd.read_excel(W8_PATH, sheet_name=W_bd_sheet, header=None).to_numpy(dtype=float)

all_bootstrap_results = []

for nama_bidang, data_file in datasets_config.items():
    if not os.path.exists(data_file):
        continue
    print(f"Menjalankan Block Bootstrap untuk Bidang: {nama_bidang}...")
    df_bidang = pd.read_excel(data_file)

    # Menyiapkan ID lokasi
    loc_order = df_bidang["Lokasi"].drop_duplicates().tolist()
    loc_to_id = {name: i for i, name in enumerate(loc_order)}
    df_bidang["loc_id"] = df_bidang["Lokasi"].map(loc_to_id)

    n_loc = df_bidang["loc_id"].nunique()
    t_years = df_bidang["Waktu"].nunique()
    n_total = len(df_bidang)

    # 1. Bootstrap untuk W_ST terpilih
    for sh in W_ST_sheets:
        W_s_base = loaded_W_ST_base[sh][:35, :35]
        W_ST_expanded = np.kron(np.eye(t_years), W_s_base)

        Wy_current = (W_ST_expanded @ df_bidang["Y"].astype(float).to_numpy().reshape(-1, 1)).ravel()
        summary = block_bootstrap_eq13(df_bidang, Wy_current, B=1000, seed=2026)
        summary["Bidang"] = nama_bidang
        summary["Weight"] = sh
        summary["Wy_type"] = f"W_ST ({n_total}x{n_total})"
        all_bootstrap_results.append(summary)

    # 2. Bootstrap untuk W_bd
    Wy_bd_current = compute_Wy_bd_dynamic(df_bidang, W_bd)
    summary = block_bootstrap_eq13(df_bidang, Wy_bd_current, B=1000, seed=2026)
    summary["Bidang"] = nama_bidang
    summary["Weight"] = "W_bd"
    summary["Wy_type"] = f"W_bd per-year ({n_loc}x{n_loc})"
    all_bootstrap_results.append(summary)

bootstrap_df = pd.DataFrame(all_bootstrap_results)
cols_boot = ["Bidang", "Weight", "Wy_type", "rho_mean", "rho_ci95", "beta_mean", "beta_ci95", "rmse_mean", "rmse_ci95", "r2_mean", "r2_ci95", "B_effective"]
bootstrap_df = bootstrap_df[cols_boot].sort_values(["Bidang", "Weight"]).reset_index(drop=True)
display(bootstrap_df)

bootstrap_df.to_excel("Hasil_Bootstrap_Gabungan_10th_3Nov.xlsx", index=False)
print("Seluruh hasil kompilasi bootstrap sukses disimpan ke 'Hasil_Bootstrap_Gabungan_10th_3Nov.xlsx'!")

Menjalankan Block Bootstrap untuk Bidang: TPT...


,Bidang,Weight,Wy_type,rho_mean,rho_ci95,beta_mean,beta_ci95,rmse_mean,rmse_ci95,r2_mean,r2_ci95,B_effective
0,TPT,W_bd,W_bd per-year (35x35),0.851825,"(0.6829111183729994, 1.026793769343165)",0.169591,"(0.06333234897084958, 0.27051299835039627)",0.693311,"(0.6165797808964681, 0.7736275518535469)",0.488777,"(0.39406447590951893, 0.5770192601410045)",1000
1,TPT,W_bdst,W_ST (350x350),0.580734,"(0.5408237000537242, 0.6165938745119184)",0.028303,"(-0.02375001698293903, 0.07964005101087963)",0.329702,"(0.2994870044797945, 0.3592936793272797)",0.883358,"(0.84183447311903, 0.9154396974244103)",1000
2,TPT,Wbq,W_ST (350x350),0.756765,"(0.6107317258522336, 0.8884194363197583)",0.161453,"(0.057424756113226984, 0.2616202809313903)",0.730129,"(0.6322221186980077, 0.8273426080873012)",0.434307,"(0.3468799810859342, 0.5283496421952363)",1000
3,TPT,Wbr,W_ST (350x350),0.728102,"(0.5896794186563472, 0.8575476093768423)",0.177345,"(0.0735647988154028, 0.2754999578354218)",0.736039,"(0.6384183978211766, 0.8313570923604853)",0.425036,"(0.338513993409445, 0.5208032378324929)",1000


Seluruh hasil kompilasi bootstrap sukses disimpan ke 'Hasil_Bootstrap_Gabungan_10th_3Nov.xlsx'!
